Import the materials dataset, feature value data, and the Python packages for processing feature values.

In [ ]:
import pandas as pd
import numpy as np

file_path = "BSPT.xlsx"
Data_df = pd.read_excel(file_path, sheet_name=0,header=0)
Feature_df = pd.read_excel(file_path, sheet_name=1, header=0,index_col=0) 

Define the function to compute the weighted average of features.

In [ ]:
def calculate_weighted_average(properties_df, probabilities_df, position):

    elements = properties_df.columns[properties_df.loc['position'] == position]
    probabilities = probabilities_df[elements]

    weighted_averages = {}
    for property_name in properties_df.index:
        if property_name == 'position':
            continue
        weighted_averages[property_name + '_' +position] = (
            properties_df.loc[property_name, elements] * probabilities
        ).sum(axis=1) / probabilities.sum(axis=1)
    
    weighted_averages_df = pd.DataFrame(weighted_averages)
    
    return weighted_averages_df

def calc_tolerance_factor(r_A, r_B, r_X=140):
    t = (r_A + r_X) / (1.414 * (r_B + r_X))
    return t

Evaluate the weighted feature average in a materials dataset.

In [ ]:
weighted_averages_df_A = calculate_weighted_average(Feature_df,Data_df,'A').drop(columns='RSC6_A')
weighted_averages_df_B = calculate_weighted_average(Feature_df,Data_df,'B').drop(columns='RSC12_B')

tolerance_factor = pd.DataFrame(calc_tolerance_factor(weighted_averages_df_A['RSC12_A'], weighted_averages_df_B['RSC6_B']),columns=['t'])

Feature_combined_df = pd.concat([Data_df.iloc[:,:2],weighted_averages_df_A,weighted_averages_df_B,tolerance_factor],axis=1)

Feature_combined_df.to_excel("Feature_data.xlsx")

Import the Python packages for computing the heatmap.

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list
import seaborn as sns
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [ ]:
Feature_hitmap = Feature_combined_df.iloc[:,2:]

correlation_matrix = Feature_hitmap.corr(method='pearson')

first_half = correlation_matrix.iloc[0:31, 0:31]
second_half = correlation_matrix.iloc[31:62, 31:62]

linkage_first = linkage(first_half, method='average', metric='euclidean')
linkage_second = linkage(second_half, method='average', metric='euclidean')

order_first = leaves_list(linkage_first)
order_second = leaves_list(linkage_second)
order_second = np.append(order_second,31)

row_order = list(order_first) + list(31 + order_second)
col_order = row_order

reordered = correlation_matrix.iloc[row_order, :].iloc[:, col_order]

Group and filter the input features

In [ ]:
distance_matrix = 1 - reordered.abs()

distance_matrix = distance_matrix.clip(lower=0)
condensed_dist = squareform(distance_matrix.values)

Z = linkage(condensed_dist, method='average')
cluster_labels = fcluster(Z, t=0.1, criterion='distance')
feature_names = reordered.columns
group_df = pd.DataFrame({'Feature': feature_names, 'Group': cluster_labels})
group_df = group_df.sort_values('Group').reset_index(drop=True)
group_df.to_excel('Feature_group.xlsx')

In [25]:
file_path = "Feature_selected.xlsx"
Feature_group_df = pd.read_excel(file_path, header=0)

In [26]:
features = Feature_group_df["Feature"].tolist()

In [28]:
selected_Feature = pd.concat([Feature_combined_df.iloc[:,:2],Feature_combined_df[Feature_combined_df.columns.intersection(features)]],axis=1)

In [30]:
selected_Feature.to_excel("DATA_A.xlsx")

In [31]:
X=selected_Feature.iloc[:,2:]
y=selected_Feature.iloc[:,:1]

Model training and output

In [32]:
from sklearn.preprocessing import StandardScaler

In [33]:
scaler = StandardScaler()
scaled_data = pd.DataFrame(scaler.fit_transform(selected_Feature.iloc[:,2:]), columns=selected_Feature.iloc[:,2:].columns)

In [34]:
import joblib
joblib.dump(scaler, 'scaler_X.pkl')

['scaler_X.pkl']

In [43]:
X=scaled_data
y=selected_Feature.iloc[:,:1]

In [40]:
selected_data_scaled = pd.concat([selected_Feature.iloc[:,:2],X],axis=1)

In [42]:
selected_data_scaled.to_excel("selected_data_scaled.xlsx")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

Y = y.values.ravel()

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.1, 
    random_state=42,
    shuffle=True
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='auto'
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    ),
    "SVR (RBF kernel)": SVR(
        kernel='rbf',
        C=1.0,
        epsilon=0.1
    ),
    "Bayesian Ridge": BayesianRidge()
}


for name, model in models.items():
    print(f"\n====== {name} ======")

    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    train_r2 = r2_score(Y_train, Y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(Y_train, Y_train_pred))

    Y_test_pred = model.predict(X_test)
    test_r2 = r2_score(Y_test, Y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(Y_test, Y_test_pred))

    print(f"  Train R² = {train_r2:.4f}")
    print(f"  Train RMSE = {train_rmse:.4f}")
    print(f"  Test R² = {test_r2:.4f}")
    print(f"  Test RMSE = {test_rmse:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np
import scipy.stats as stats

Y = y.values.ravel()


X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.1, 
    random_state=42,
    shuffle=True
)


param_dist = {
    "n_estimators": stats.randint(100, 800),        
    "learning_rate": stats.uniform(0.01, 0.2),      
    "max_depth": stats.randint(2, 8),               
    "subsample": stats.uniform(0.6, 0.4),            
    "colsample_bytree": stats.uniform(0.6, 0.4),     
    "reg_alpha": stats.uniform(0, 1.0),              
    "reg_lambda": stats.uniform(0, 1.0)              
}


xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    tree_method='auto'
)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=100,                  
    scoring='r2', 
    cv=10,                       
    verbose=2,                  
    random_state=42,
    n_jobs=-1,
    refit=True,                             
    return_train_score=True                 #
)

random_search.fit(X_train, Y_train)
print(random_search.best_params_)

best_model = random_search.best_estimator_

Y_train_pred = best_model.predict(X_train)
train_r2 = r2_score(Y_train, Y_train_pred)
train_rmse = np.sqrt(mean_squared_error(Y_train, Y_train_pred))

Y_test_pred = best_model.predict(X_test)
test_r2 = r2_score(Y_test, Y_test_pred)
test_rmse = np.sqrt(mean_squared_error(Y_test, Y_test_pred))

print("\n====== Tuned XGBoost (After RandomizedSearchCV) ======")
print(f"  Train R² = {train_r2:.4f}")
print(f"  Train RMSE = {train_rmse:.4f}")
print(f"  Test R² = {test_r2:.4f}")
print(f"  Test RMSE = {test_rmse:.4f}")

train_results = pd.DataFrame({
    'Actual': Y_train,
    'Predicted': Y_train_pred
})

test_results = pd.DataFrame({
    'Actual': Y_test,
    'Predicted': Y_test_pred
})

In [47]:
train_results = pd.DataFrame({
    'Actual': Y_train,
    'Predicted': Y_train_pred
})

test_results = pd.DataFrame({
    'Actual': Y_test,
    'Predicted': Y_test_pred
})

In [48]:
with pd.ExcelWriter('xgboost_predictions.xlsx') as writer:
    train_results.to_excel(writer, sheet_name='Train', index=False)
    test_results.to_excel(writer, sheet_name='Test', index=False)

In [ ]:
import joblib
joblib.dump(best_model, 'best_xgb_model.pkl')

In [49]:
X=scaled_data
y=selected_Feature.iloc[:,1:2]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


Y = y.values.ravel()

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.1, 
    random_state=42,
    shuffle=True
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=5,
        min_samples_leaf=3,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='auto'
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=4,
        random_state=42
    ),
    "SVR (RBF kernel)": SVR(
        kernel='rbf',
        C=1.0,
        epsilon=0.1
    ),
    "Bayesian Ridge": BayesianRidge()
}


for name, model in models.items():
    print(f"\n====== {name} ======")

    model.fit(X_train, Y_train)

    Y_train_pred = model.predict(X_train)
    train_r2 = r2_score(Y_train, Y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(Y_train, Y_train_pred))

    Y_test_pred = model.predict(X_test)
    test_r2 = r2_score(Y_test, Y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(Y_test, Y_test_pred))

    print(f"  Train R² = {train_r2:.4f}")
    print(f"  Train RMSE = {train_rmse:.4f}")
    print(f"  Test R² = {test_r2:.4f}")
    print(f"  Test RMSE = {test_rmse:.4f}")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import r2_score
import numpy as np
import scipy.stats as stats

Y = y.values.ravel()

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.1, 
    random_state=42,
    shuffle=True
)

param_dist = {
    "alpha_1": stats.loguniform(1e-8, 1e-1),    
    "alpha_2": stats.loguniform(1e-8, 1e-1),    
    "lambda_1": stats.loguniform(1e-8, 1e-1),   
    "lambda_2": stats.loguniform(1e-8, 1e-1),    
    "tol": stats.loguniform(1e-6, 1e-3)          
}

br = BayesianRidge()

random_search = RandomizedSearchCV(
    estimator=br,
    param_distributions=param_dist,
    n_iter=100,                      
    scoring='r2',                   
    cv=10,                           
    verbose=2,                       
    random_state=42,
    n_jobs=-1,
    refit=True,
    return_train_score=True
)

random_search.fit(X_train, Y_train)

print(random_search.best_params_)

best_model = random_search.best_estimator_

Y_train_pred = best_model.predict(X_train)
train_r2 = r2_score(Y_train, Y_train_pred)
train_maer = np.mean(np.abs((Y_train - Y_train_pred) / (Y_train + 1e-8)))

Y_test_pred = best_model.predict(X_test)
test_r2 = r2_score(Y_test, Y_test_pred)
test_maer = np.mean(np.abs((Y_test - Y_test_pred) / (Y_test + 1e-8)))

print("\n====== Tuned Bayesian Ridge (After RandomizedSearchCV) ======")
print(f"  Train R² = {train_r2:.4f}")
print(f"  Train RMSE = {train_rmse:.4f}")
print(f"  Test R² = {test_r2:.4f}")
print(f"  Test RMSE = {test_rmse:.4f}")

In [53]:
train_results = pd.DataFrame({
    'Actual': Y_train,
    'Predicted': Y_train_pred
})

test_results = pd.DataFrame({
    'Actual': Y_test,
    'Predicted': Y_test_pred
})

In [54]:
with pd.ExcelWriter('BR_predictions.xlsx') as writer:
    train_results.to_excel(writer, sheet_name='Train', index=False)
    test_results.to_excel(writer, sheet_name='Test', index=False)

In [286]:
joblib.dump(best_model, 'best_BR_model.pkl')

['best_BR_model.pkl']